In [1]:
import os
import pandas as pd

import requests
import googlemaps
import json

from geopy.distance import distance
from geopy.distance import geodesic

from dotenv import load_dotenv

from tqdm import tqdm

from math import ceil

# MongoDB
from pymongo import MongoClient 
from pymongo.errors import BulkWriteError

In [2]:
# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [3]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [4]:
centers_cache = {}
bad_locations = {}

In [5]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

In [6]:
# Google Maps API handler
def getCenterCoords(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}", components={"country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            
            return [latitude, longitude]
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return [None, None]
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return [None, None]

In [7]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}", components={"country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            coords = [longitude, latitude]

            # Get the city from the address components
            address_components = geocode_result[0]['address_components']
            city = None
            state = None
            for component in address_components:
                if 'locality' in component['types']:
                    city = component['long_name']
                elif 'administrative_area_level_1' in component['types']:
                    state = component['long_name']
            
            location_type = geocode_result[0]["geometry"]["location_type"]
            if location_type == "APPROXIMATE":
                bad_locations[location] = "Approximate location"
                return None, None, None
            
            if city:
                if city in centers_cache:
                    if centers_cache[city] == coords:
                        bad_locations[location] = "Same coordinates as city"
                        return None, None, None
                else:
                    city_coords = getCenterCoords(city)
                    centers_cache[city] = city_coords
                    if city_coords == coords:
                        bad_locations[location] = "Same coordinates as city"
                        return None, None, None
            
            if state:
                if state in centers_cache:
                    if centers_cache[state] == coords:
                        bad_locations[location] = "Same coordinates as state"
                        return None, None, None
                else:
                    state_coords = getCenterCoords(state)
                    centers_cache[state] = state_coords
                    if state_coords == coords:
                        bad_locations[location] = "Same coordinates as state"
                        return None, None, None
            
            return longitude, latitude, city
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return None, None, None
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return None, None, None

In [8]:
known_locations_path = "./../data_prod/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [9]:
# Get the coordinates of the location
def getCoordinates(location): 
    try:
        if (location == None or len(location) == 0): return None  
        # Only get coordinates if the location is not already known
        if (location in known_locations):
            longitude, latitude = known_locations[location]["coordinates"]
        else:
            # Get coordinates and save to cache
            longitude, latitude, city = callGoogleMapsAPI(location)
            if (longitude is None or latitude is None): return None
            
            known_locations[location] = {"coordinates": [longitude, latitude], "city": city, "state": None, "tract": None, "county": None}
            save_cache_to_file(known_locations, known_locations_path)

        return [longitude, latitude]
    except Exception as error:
        print(f"[ERROR] Error getting coordinates for {location}: {error}")
        return None

In [10]:
def getAllCoordinates(locations):
    if (locations == None or len(locations) == 0): return None, None

    try: 
        good_locs = []
        good_coords = []
        for location in locations:
            if location is None or len(location) == 0: continue
            coordinates = getCoordinates(location)
            if coordinates is not None:
                good_locs.append(location)
                good_coords.append(coordinates)

        if len(good_locs) == 0: return None, None
        return good_locs, good_coords
        
    except Exception as error:
        print(f"[ERROR] Error processing locations: {locations}, Error: {error}")
        return None, None


In [11]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']
            state = results['result']['geographies']['2020 Census Blocks'][0]['STATE']
            
            return tract, county, state
        except IndexError:
            print("[ERROR] Unable to retrieve census geography for: " + location)
        except KeyError:
            print("[ERROR] Location is outside of the United States: " + location)
        except Exception as error:
            print(f"[ERROR] Error retrieving census geography for: {location}, Error: {error}")

    print("[ERROR] API call failed for: " + location + " with coordinates" + str(coordinates))
    return None, None, None  # Return this if API call failed or no tracts found

In [12]:
# Get the census tract and county of the location
def geocode(location, coordinates):
    if (location is None or len(location) == 0): return None, None, None, None  

    if (coordinates is None or len(coordinates) == 0
        or coordinates[0] is None or coordinates[1] is None): return None, None, None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    State = known_locations[location]["state"]
    City = known_locations[location]["city"]

    if (Tract is None or County is None or State is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County, State = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        known_locations[location]["state"] = State
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County, State, City

In [13]:
def getAllGeocodes(locations, coordinates):
    tracts = []
    counties = []
    states = []
    cities = []

    if (locations == None or len(locations) == 0): return None, None, None, None
    if (coordinates == None or len(coordinates) == 0): return None, None, None, None
    
    for i, location in enumerate(locations):
        try:
            Tract, County, State, City = geocode(location, coordinates[i])
        except Exception as error:
            print(f"[ERROR] Error geocoding location: {location}, Error: {error}")
            Tract, County, State, City = None, None, None, None
            
        tracts.append(Tract)
        counties.append(County)
        states.append(State)
        cities.append(City)


    return tracts, counties, states, cities

In [14]:
neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800", 
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
}

In [15]:
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = "locations_test"
mongo_client = MongoClient(mongo_uri)

In [16]:
# Get the collection from the database
def get_collection(db_prod, collection_name):
	collection_list = db_prod.list_collection_names()

	# Initialize the collection if it doesn't exist
	if collection_name not in collection_list:
		db_prod.create_collection(collection_name)
		print(f"[INFO] Collection '{collection_name}' created.")

	return db_prod[collection_name]

# Get all neighborhoods from the database for geocoding
def get_neighborhoods(client):

    db_prod = client[mongo_db_name]

    neighborhood_collection = get_collection(db_prod, "neighborhood_data")
    tract_to_neighborhood = {}
    neighborhood_to_tracts = {}
    
    neighborhoods = neighborhood_collection.find()

    # Populate the dictionary with tract-to-neighborhood mappings
    for neighborhood in neighborhoods:
        neighborhood_name = neighborhood.get('value')
        tracts = neighborhood.get('tracts', [])
        
        for tract in tracts:
            tract_to_neighborhood[tract] = neighborhood_name

        if neighborhood_name not in neighborhood_to_tracts:
            neighborhood_to_tracts[neighborhood_name] = tracts

    return tract_to_neighborhood, neighborhood_to_tracts

In [17]:
try:
    tract_map, neigh_map = get_neighborhoods(mongo_client)
except Exception as error:
    print(f"[ERROR] Error getting neighborhoods from database: {error}")
    tract_map_path = "./data_prod/tract_map.json"
    tract_map = load_cache(tract_map_path)
    neigh_map = neigh_tract_dict
    if (tract_map == {}):
        for neigh, tracts in neigh_tract_dict.items():
            for tract in tracts:
                tract_map[tract] = neigh

        save_cache_to_file(tract_map, tract_map_path)

In [18]:
def create_neighborhood(tract, neighborhood):
    if neighborhood not in neigh_map:
        neigh_map[neighborhood] = [tract]
    else:
        neigh_map[neighborhood].append(tract)
    tract_map[tract] = neighborhood


In [19]:
def getAllNeighborhoods(articles):
    tracts = articles['tracts']

    neighborhoods = []

    if (articles['locations'] == None or len(articles['locations']) == 0): return None
    if (articles['coordinates'] == None or len(articles['coordinates']) == 0): return None
    if (tracts == None or len(tracts) == 0): return None

    for i, tract in enumerate(tracts):
        if (tract == None): 
            neighborhoods.append("No Neighborhood")
            continue

        neighborhood = tract_map.get(tract)
        if neighborhood is not None:
            neighborhoods.append(neighborhood)
        else:
            city = articles['cities'][i]
            if city is None:
                neighborhoods.append("Unknown Neighborhood")
            else:
                create_neighborhood(tract, city)
                neighborhoods.append(city)

    return neighborhoods

In [20]:
# Get census demographics for any given article
def get_census_demographics(year, dsource, dname, tract, county, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    census_url = f"{base_url}?get={cols}&for=tract:{tract}&in=county:{county}&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()

    return census_response_json

def get_city_demographics(year, dsource, dname, city, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    # Note: Adjust 'for' and 'in' parameters based on city-level geography
    census_url = f"{base_url}?get={cols}&for=place:*&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()
    
    # Filter results to find the specific city
    city_demographics = [
        item for item in census_response_json[1:]
        if city in item[0]  # Assuming the city name is in the first column of the results
    ]
    columns = cols.split(",")
    city_demographics = [columns, city_demographics[0]]
    return city_demographics

def update_demographics(tract_collection, tract, county, state, city=None):
    try:
        if city:
            census_data = get_city_demographics("2020", "dec", "pl", city, state)
        else:
            census_data = get_census_demographics("2020", "dec", "pl", tract, county, state)
        
        if not census_data or len(census_data) < 2:
            raise ValueError("Census data is missing or malformed.")
        
        headers = census_data[0]  # Headers
        values = census_data[1]   # Data values
        data = dict(zip(headers, values))

        county_name = data.get('NAME', "")
        geoid_tract = f"{state}{county}{tract}"

        # Prepare the update document
        update_doc = {
            'demographics.p2_001n': str(data.get('P2_001N', 0)),
            'demographics.p2_002n': str(data.get('P2_002N', 0)),
            'demographics.p2_003n': str(data.get('P2_003N', 0)),
            'demographics.p2_004n': str(data.get('P2_004N', 0)),
            'demographics.p2_005n': str(data.get('P2_005N', 0)),
            'demographics.p2_006n': str(data.get('P2_006N', 0)),
            'demographics.p2_007n': str(data.get('P2_007N', 0)),
            'demographics.p2_008n': str(data.get('P2_008N', 0)),
            'demographics.p2_009n': str(data.get('P2_009N', 0)),
            'demographics.p2_010n': str(data.get('P2_010N', 0)),
            'county_name': county_name,
            'geoid_tract': geoid_tract
        }
        
        # Update MongoDB document
        tract_collection.update_one(
            {'tract': tract},
            {'$set': update_doc}
        )

    except Exception as error:
        print(f"[ERROR] Error getting census data for tract {tract}: {error}")
        tract_collection.update_one(
            {'tract': tract},
            {'$set': {"canFind": False}}
        )
        return


In [21]:
def update_tracts(tract_collection, tract, neighborhood, county, state, city, article, location):    
    # Check if the tract document exists and has demographics data
    empty_doc = {
        'tract': tract,
        'state': state,
        'county': county,
        'city': city,
        'neighborhood': neighborhood,
        'county_name': "",
        'geoid_tract': "",
        'demographics': {},
        'articles': [article],
        'locations': [location],
        'canFind': True
    }
    tract_collection.insert_one(empty_doc)
    update_demographics(tract_collection, tract, county, state)

In [22]:
def create_gen_tract(tract_collection, tract, neighborhood, county, state, city, article, location):    
    # Check if the tract document exists and has demographics data
    empty_doc = {
        'tract': tract,
        'state': state,
        'county': "",
        'city': city,
        'neighborhood': neighborhood,
        'county_name': "",
        'geoid_tract': "",
        'demographics': {},
        'articles': [article],
        'locations': [location],
        'canFind': True
    }
    tract_collection.insert_one(empty_doc)
    update_demographics(tract_collection, tract, county, state, city)

In [23]:
def removeRepeatedCoords(row):
    coordinates = row['coordinates']
    if coordinates is None or not isinstance(coordinates, list):
        return row
    
    coord_counts = {}
    indexes_to_remove = set()
    
    for i, coord in enumerate(coordinates):
        if coord is None or coord[0] is None or coord[1] is None:
            print(f"[WARNING] Removing unprocessed location: {row['locations'][i]}")
            indexes_to_remove.add(i)
            continue
        elif row['tracts'][i] is None:
            print(f"[WARNING] Removing location with no tract: {row['locations'][i]}")
            indexes_to_remove.add(i)
            continue
        
        coord_tuple = tuple(coord) 
        if coord_tuple in coord_counts:
            indexes_to_remove.add(i)
        else:
            coord_counts[coord_tuple] = i

    # Remove duplicates from each column
    for column in ['locations', 'coordinates', 'tracts', 'counties', 'states', 'cities', 'neighborhoods']:
        if isinstance(row[column], list):
            row[column] = [v for i, v in enumerate(row[column]) if i not in indexes_to_remove]
    

    return row


In [24]:
def geolocate_articles(df):
    """
    Processes the dataaframe given by func. Does Entity Recognition and Geolocation on articles.
    
    Parameters
    ----
    df: The pandas dataframe that geolocation is being done on.

    Returns
    ---- 
    Returns a Dataframe of geolocated articles
    """
    try: 
        df[["locations", "coordinates"]] = df["locations"].apply(lambda row: pd.Series(getAllCoordinates(row)))

        # Geocode the Coordinates (Get the Tract and County)
        df[['tracts', 'counties', 'states', 'cities']] = df.apply(lambda row: pd.Series(getAllGeocodes(row['locations'], row['coordinates'])), axis=1)

        # Get the Neighborhoods
        df["neighborhoods"] = df.apply(getAllNeighborhoods, axis=1)

        for column in ['locations', 'coordinates', 'tracts', 'counties', 'states', 'cities', 'neighborhoods']:
            df[column] = df[column].apply(lambda x: None if (x is None or len(x) == 0) else x)

        # Drop the rows that are missing information
        df = df.dropna(subset=["locations", "coordinates", "tracts", "counties", 'states', 'cities', "neighborhoods"]).reset_index(drop=True) # Clean the rows that are missing information

        # Remove repeated coordinates
        df = df.apply(removeRepeatedCoords, axis=1)

        return df
    except Exception as e: 
        print(f"[Fatal Error] geolocate_articles() ran into an Error! Data is not saved!\nRaw Error:{e}")
        raise Exception(f"FATAL ERROR {e}")
    return

In [25]:
# Pack articles and send to MongoDB
def pack_articles(db_prod, df):
	try:
		article_payload = df.to_dict(orient='records')

		articles_collection = get_collection(db_prod, "articles_data")

		try: 
			articles_collection.insert_many(article_payload, ordered=False)
		except BulkWriteError as bwe:
			# Handle duplicate key errors
			write_errors = bwe.details.get('writeErrors', [])
			duplicates = [error['op'] for error in write_errors if error['code'] == 11000]
			if duplicates:
				print(f"[WARNING] Skipped {len(duplicates)} duplicate articles.")
			else:
				raise bwe
		return
	
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Article Data\nError: {err}")
	return

# Pack the neighborhood data and send to MongoDB
def pack_neighborhoods(db_prod, df):
	try:
		neigh_collection = get_collection(db_prod, "neighborhood_data")

		# TODO: Delete this after neighborhood data is updated
		# for neighborhood in neigh_tract_dict.keys():
		# 	neigh_collection.update_one(
		# 		{'value': neighborhood},
		# 		{'$setOnInsert': {'tracts': neigh_tract_dict[neighborhood]}},
		# 		upsert = True # Creates a new document of it if it doesn't exist
		# 	)
		# print("[INFO] Neighborhoods Collection Successfully Populated!")

		# Save all new neighborhoods with associated tracts and articles
		for n, neighborhoods in enumerate(df['neighborhoods']):
			for i, neighborhood in enumerate(neighborhoods):				
				# More convoluted than it should be. There's a bug with addToSet so this is a workaround
				current_doc = neigh_collection.find_one({'value': neighborhood})
				update_data = {}
				
				if current_doc:
					if df["_id"][n] not in current_doc.get('articles', []):
						update_data['articles'] = df["_id"][n]
					if df['tracts'][n][i] not in current_doc.get('tracts', []):
						update_data['tracts'] = df['tracts'][n][i]
					if df['locations'][n][i] not in current_doc.get('locations', []):
						update_data['locations'] = df['locations'][n][i]
				else:
					# Initialize values
					update_data['articles'] = df["_id"][n]
					update_data['tracts'] = df['tracts'][n][i]
					update_data['locations'] = df['locations'][n][i]

				# Update the document if there's anything to update
				if update_data:
					neigh_collection.update_one(
						{'value': neighborhood},
						{
							'$push': update_data
						}, upsert=True
					)
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Neighborhood Data\nError: {err}")
	return

# Pack the topics data and send to MongoDB
def pack_topics(db_prod, df):
	try:
		topic_collection = get_collection(db_prod, "topics_data")

		# Save all new topics with associated articles
		for n, topic in enumerate(df["openai_labels"]):
			topic_collection.update_one(
				{'value': topic},
				{'$addToSet': {'articles': df["_id"][n]}},
				upsert = True 
			)          
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Topics Data\nError: {err}")
	return

# Pack the tracts data and send to MongoDB
def pack_tracts(db_prod, df):
	try:
		tract_collection = get_collection(db_prod, "tracts_data")

		# Save all new tracts with associated articles and neighborhoods
		for n, tracts in enumerate(df['tracts']):
			for i, tract in enumerate(tracts):
				if tract_collection.find_one({'tract': tract}):
					tract_collection.update_one(
					{'tract': tract},
					{
					 '$push': {'articles': df['_id'][n]},
	  				 '$addToSet': {'locations': df['locations'][n][i]},
					}, upsert=True
    				) 
				else:
					update_tracts(tract_collection, tract, df["neighborhoods"][n][i], df["counties"][n][i], df["states"][n][i], df["cities"][n][i], df['_id'][n], df['locations'][n][i]) 
				
				# Create and update global tract for neighborhood/city
				gen_tract = df["neighborhoods"][n][i]
				if tract_collection.find_one({'tract': gen_tract}):
					tract_collection.update_one(
					{'tract': gen_tract},
					{
					 '$push': {'articles': df['_id'][n]},
	  				 '$addToSet': {'locations': df['locations'][n][i]},
					}, upsert=True
					)
				else:
					create_gen_tract(tract_collection, gen_tract, gen_tract, df["counties"][n][i], df["states"][n][i], df["cities"][n][i], df['_id'][n], df['locations'][n][i])

			  
	except Exception as err:
		raise Exception(f"[ERROR] Error in sending Tracts Data\nError: {err}")
	return

def best_location(locations):
	if len(locations) == 1: return locations[0]
	
	# Remove some/(most?) abbreviations
	locations = [location for location in locations if len(location) > 3]
	if len(locations) == 1: return locations[0]

	sorted_locations = sorted(locations, key=len)

	def find_substring(sorted_list):
		for i in range(len(sorted_list)):
			substring = sorted_list[i]
			if len(substring.split(" ")) == 1:
				continue
			# Check if this substring is in at least some of the others
			count = sum(1 for other in sorted_list if substring in other and other != substring)
			if count > 0:
				return substring
		return None

	# Get the result
	result = find_substring(sorted_locations)
	if result:
		return result
	
	if len(sorted_locations[0].split(" ")) > 1:
		return sorted_locations[0]
	else:
		return sorted_locations[1]
    

# Pack the locations data and send to MongoDB
def pack_locations(db_prod, df):
	try:
		location_collection = get_collection(db_prod, "locations_data")

		# Save all new locations with associated articles
		for n, locations in enumerate(df["locations"]):
			for i, location in enumerate(locations):
				coordinates = df["coordinates"][n][i]

				# Try to find the location document based on coordinates
				location_doc = location_collection.find_one({'coordinates': coordinates})
				if location_doc:
					all_locations = location_doc.get('all_locations', [])
					if location not in all_locations:
						all_locations.append(location)
						best_loc = best_location(all_locations)
						current_loc = location_doc.get('value', None)
						
						# Update location document with the new best location if needed
						update_fields = {
							'$addToSet': {
								'articles': df["_id"][n],
								'all_locations': location
							}
						}
						if best_loc != current_loc:
							update_fields['$set'] = {'value': best_loc}

						location_collection.update_one(
							{'coordinates': coordinates},
							update_fields
						)
					else:
						# Only add the article if the location already exists
						location_collection.update_one(
							{'coordinates': coordinates},
							{'$addToSet': {'articles': df["_id"][n]}}
						)
				else:
					# Insert a new document with the specified fields
					location_collection.insert_one({
						'coordinates': coordinates,
						'articles': [df["_id"][n]],
						'all_locations': [location],
						'value': location,
						'neighborhood': df["neighborhoods"][n][i],
						'tract': df["tracts"][n][i],
						'city': df["cities"][n][i],
						'state': df["states"][n][i]
					})

          
	except Exception as err:
		raise Exception(f"[ERROR]  Error in sending Locations Data\nError: {err}")
	return


In [26]:
# Handle the entire process of sending data to production
def send_to_production(client, db_name, df):
	try:
		db_prod = client[db_name]

		# Pack and send all articles
		pack_articles(db_prod, df)
		pack_neighborhoods(db_prod, df)
		pack_topics(db_prod, df)
		pack_tracts(db_prod, df)
		pack_locations(db_prod, df)
		print("[INFO] Data Successfully Sent to Production!")

	except Exception as err:
		print(f"[ERROR] Error in sending data to MongoDB Prod DB\nError: {err}")
		raise Exception("Fatal Error in sending to production")
	return

In [27]:
articles_collection = get_collection(mongo_client["naacp_db"], "articles_data")
full_df = pd.DataFrame(list(articles_collection.find()))

In [28]:
start_point = 0
end_point = len(full_df)
# end_point = 3
batch_size = 100

In [29]:
# Process articles in batches of 100
def process_articles(articles_df):
    results_df = pd.DataFrame()
    batch_count = (start_point // batch_size)
    batch_total = batch_count + ceil((end_point - start_point)/ batch_size)
    batch_count += 1
    total_count = 0

    for batch in range(start_point, end_point, batch_size):
        print(f"[INFO] Processing batch {batch_count} of {batch_total}")
        articles = articles_df[batch:batch + batch_size].copy()
        
        # Conduct Entity Recognition
        print(f"[INFO] Processing through Geolocation Pipeline")
        processing_df = geolocate_articles(articles)

        total_count += processing_df.shape[0]

        print("[INFO] Sending Data to MongoDB Production")
        
        send_to_production(mongo_client, mongo_db_name, processing_df)
        
        batch_count += 1
        print(f"[INFO] Batch Complete! Recognized and uploaded {processing_df.shape[0]} articles \n")


    print(f"[INFO] Inference Pipeline Complete! Total Articles Processed: {total_count}")
    return results_df


In [30]:
process_articles(full_df)

[INFO] Processing batch 1 of 105
[INFO] Processing through Geolocation Pipeline
[INFO] Sending Data to MongoDB Production
[INFO] Collection 'articles_data' created.
[INFO] Collection 'topics_data' created.
[INFO] Collection 'tracts_data' created.
[ERROR] Error getting census data for tract Middleton: list index out of range
[ERROR] Error getting census data for tract Unknown Neighborhood: Expecting value: line 1 column 1 (char 0)
[ERROR] Error getting census data for tract Hamilton: list index out of range
[ERROR] Error getting census data for tract Easton: list index out of range
[ERROR] Error getting census data for tract Saint Paul: list index out of range
[INFO] Collection 'locations_data' created.
[INFO] Data Successfully Sent to Production!
[INFO] Batch Complete! Recognized and uploaded 63 articles 

[INFO] Processing batch 2 of 105
[INFO] Processing through Geolocation Pipeline
[INFO] Sending Data to MongoDB Production
[ERROR] Error getting census data for tract South Hadley: li

""
